# MCP Mini Protocol
# 0. 介绍

**研究背景**：Agent 要读取数据或执行操作，必须连接由外部服务提供的资源和工具。这些服务通常由不同团队、不同系统实现，因此外层程序需要一种统一方式，让 Agent 知道服务提供什么能力、调用时要传什么参数，以及调用后返回什么结果。

**现存问题**：如果每个工具都使用自己的请求格式和硬编码适配逻辑，新增或更换服务时就必须重新编写连接代码。这样，即使大模型选对了工具和参数，请求也可能因为方法名、参数结构或返回格式不一致而无法执行；没有统一的请求编号和错误结构时，外层程序还难以判断哪次调用失败、为什么失败。

**解决方案**：本 Notebook 将实现一个极简的 MCP Mini Protocol，采用当前 MCP 标准的 `Host-Client-Server` 分工和 `JSON-RPC 2.0` 消息格式，用统一的 `tools/list`、`tools/call` 和 `resources/read` 完成能力发现、工具调用与资源读取，并通过请求 ID 和结构化错误检查每次响应。然后用同一份真实 API 决定进行对比：基线版本因私有协议格式不一致而失败，改进版本通过统一协议完成相同任务，从而直观看到标准协议如何减少工具接入耦合，并让调用过程可以验证和诊断。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 准备资源和目标工具
MCP 把可读取的数据称为资源，把可执行的操作称为工具。下面准备一个保存两个整数的资源，并指定稍后要调用的加法工具；基线版本和改进版本都会使用这份相同数据。

In [2]:
# 固定两种协议共同使用的资源和目标工具
resource_uri = "numbers://example"
resource_data = {"a": 20, "b": 22}
target_tool_name = "add"

print(f"资源：{resource_uri} -> {resource_data}")
print(f"目标工具：{target_tool_name}")

资源：numbers://example -> {'a': 20, 'b': 22}
目标工具：add


输出显示资源中保存了 `20` 和 `22`，目标工具是 `add`。此时只是固定了任务数据，还没有读取资源或执行加法；下一步让大模型用统一格式说明它准备连接哪个资源和工具。

## 2.2 说明大模型需要作出的决定
协议负责传输请求，大模型负责选择要使用的资源和工具。下面定义一个简单的决定表单，让大模型只填写资源地址和工具名称，避免把协议封包细节交给大模型生成。

In [3]:
# 定义真实 API 返回决定时使用的结构化表单
decision_tool = {
    "type": "function",
    "function": {
        "name": "choose_mcp_action",
        "description": "选择需要读取的资源和随后调用的工具",
        "parameters": {
            "type": "object",
            "properties": {
                "resource_uri": {"type": "string"},
                "tool_name": {"type": "string"},
            },
            "required": ["resource_uri", "tool_name"],
        },
    },
}

print(f"决定表单：{decision_tool['function']['name']}")

决定表单：choose_mcp_action


输出显示 `choose_mcp_action` 表单已经准备好。它只记录大模型的选择，不读取资源，也不执行工具；这些动作将由后续章节中的协议客户端完成。

## 2.3 写出共同任务
为了只比较协议差异，基线版本和改进版本必须复用同一次大模型决定。下面要求大模型读取指定资源，并把资源中的两个整数交给加法工具。

In [4]:
# 写出稍后发送给真实大模型的共同任务
messages = [
    {
        "role": "system",
        "content": "你负责选择完成任务所需的资源和工具。",
    },
    {
        "role": "user",
        "content": (
            f"读取 {resource_uri}，"
            f"再调用 {target_tool_name} 计算其中两个整数的和。"
        ),
    },
]

print(f"任务：{messages[-1]['content']}")

任务：读取 numbers://example，再调用 add 计算其中两个整数的和。


输出显示任务已经固定：先读取 `numbers://example`，再调用 `add`。下一章将把这份任务和决定表单发送给真实大模型，并保存模型返回的资源地址和工具名称。

# 3. 获取并验证 API 响应
## 3.1 获取真实大模型的决定
大模型只负责决定使用哪个资源和工具，协议客户端负责执行这个决定。下面把上一章的任务和决定表单发送给真实 API，并记录等待响应所用的时间。

In [5]:
import time

# 调用一次真实 API，并记录模型响应时间
started_at = time.perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=[decision_tool],
    tool_choice={
        "type": "function",
        "function": {"name": "choose_mcp_action"},
    },
)
latency_ms = round((time.perf_counter() - started_at) * 1000)

print("真实 API 已返回决定")

真实 API 已返回决定


输出说明真实 API 已经完成一次请求。此时外部资源和加法工具仍未执行；下一步只读取模型返回的结构化决定。

## 3.2 读取结构化决定
真实 API 返回的是一个工具调用对象。下面从中取出函数参数，并转换成普通字典，供基线版本和改进版本共同使用。

In [6]:
import json

# 读取真实大模型返回的资源地址和工具名称
tool_call = response.choices[0].message.tool_calls[0]
decision = json.loads(tool_call.function.arguments)
chosen_resource_uri = decision["resource_uri"]
chosen_tool_name = decision["tool_name"]

print(f"模型选择资源：{chosen_resource_uri}")
print(f"模型选择工具：{chosen_tool_name}")

模型选择资源：numbers://example
模型选择工具：add


输出显示大模型选择了任务指定的资源和工具。这个决定只生成一次，后面的两种协议实现都会直接复用它，因此实验差异只来自外层协议。

## 3.3 查看本次 API 调用
为了知道这次决定由哪个模型生成、消耗了多少 token，以及等待了多久，下面把 API 返回的运行信息集中打印出来。

In [7]:
# 汇总本次真实 API 调用的可观测信息
api_record = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": response.usage.prompt_tokens,
    "output_tokens": response.usage.completion_tokens,
    "total_tokens": response.usage.total_tokens,
    "latency_ms": latency_ms,
    "stop_reason": response.choices[0].finish_reason,
}

print(json.dumps(api_record, ensure_ascii=False, indent=2))

{
  "provider": "openai",
  "model": "LongCat-2.0",
  "input_tokens": 183,
  "output_tokens": 98,
  "total_tokens": 281,
  "latency_ms": 3904,
  "stop_reason": "tool_calls"
}


输出记录了本次真实 API 调用的 provider、model、token、延迟和停止原因。大模型的决定与运行成本已经固定，下一章将定义不使用统一协议的基线组件。

# 4. 定义基线组件 *
## 4.1 定义私有客户端格式
没有统一协议时，客户端通常会自行规定请求格式。下面的客户端把动作放在 `action`，把具体数据放在 `payload`；它不知道外部服务实际使用什么字段。

In [8]:
# 按客户端自己的约定创建私有请求
def build_private_request(action, payload):
    return {"action": action, "payload": payload}


private_request_example = build_private_request(
    "read",
    {"uri": chosen_resource_uri},
)
print(f"客户端请求：{private_request_example}")

客户端请求：{'action': 'read', 'payload': {'uri': 'numbers://example'}}


输出显示客户端只会生成 `action/payload` 结构。这个请求本身没有问题，但外部服务必须恰好理解相同字段才能执行；下一步查看资源服务自己的格式。

## 4.2 定义私有资源服务
资源服务由另一套代码实现，它没有统一协议可参考，因此直接从请求顶层读取 `address`。下面定义这个服务及其可读取的数据。

In [9]:
private_resources = {resource_uri: resource_data}


# 按资源服务自己的字段读取数据
def private_read(request):
    address = request.get("address")
    return private_resources.get(address)


print("资源服务读取字段：address")

资源服务读取字段：address


输出显示资源服务只读取顶层的 `address`，而客户端把地址放在 `payload.uri`。双方描述的是同一个地址，却没有共同的数据结构；下一步查看工具服务的另一套格式。

## 4.3 定义私有工具服务
加法工具也使用自己的请求格式，它只从顶层的 `numbers` 列表取得输入。下面定义这个最小工具服务。

In [10]:
# 按工具服务自己的字段执行加法
def private_add(request):
    numbers = request.get("numbers", [])
    return sum(numbers)


print("工具服务读取字段：numbers")

工具服务读取字段：numbers


输出显示工具服务只读取顶层的 `numbers`，这与客户端的 `action/payload` 仍不一致。至此，基线中的客户端、资源服务和工具服务都已定义；下一章将把同一次大模型决定交给它们，直接观察私有格式混用造成的结果。

# 5. 展示基线故障 *
## 5.1 发送资源请求
先把真实大模型选择的资源地址交给私有客户端。客户端会生成自己的 `action/payload` 请求，再把这份请求原样发送给资源服务。

In [11]:
# 使用真实大模型选择的地址读取资源
baseline_resource_request = build_private_request(
    "read",
    {"uri": chosen_resource_uri},
)
baseline_resource_result = private_read(baseline_resource_request)

print(f"资源请求：{baseline_resource_request}")
print(f"资源返回：{baseline_resource_result}")

资源请求：{'action': 'read', 'payload': {'uri': 'numbers://example'}}
资源返回：None


资源返回了 `None`。原因很直接：客户端把地址放在 `payload.uri`，资源服务却只读取顶层的 `address`。大模型选对了地址，但私有格式没有把地址送到服务能够读取的位置。

## 5.2 发送工具请求
接着把资源返回值和真实大模型选择的工具名称交给同一个私有客户端。客户端仍然使用 `action/payload`，而加法服务只读取顶层的 `numbers`。

In [12]:
# 把资源结果交给真实大模型选择的工具
baseline_tool_request = build_private_request(
    "call",
    {
        "name": chosen_tool_name,
        "arguments": baseline_resource_result,
    },
)
baseline_tool_result = private_add(baseline_tool_request)

print(f"工具请求：{baseline_tool_request}")
print(f"工具返回：{baseline_tool_result}")

工具请求：{'action': 'call', 'payload': {'name': 'add', 'arguments': None}}
工具返回：0


工具返回了 `0`。加法服务没有在请求顶层找到 `numbers`，因此没有取得原本的 `20` 和 `22`。资源地址和工具名称都由真实大模型正确选择，数据仍在两次私有格式转换中丢失。

## 5.3 展示最终结果
最后把资源中的正确答案与基线实际返回值放在一起。这里只展示结果，不改变任何请求或服务。

In [13]:
# 保存后续对照实验需要的基线结果
expected_result = sum(resource_data.values())
baseline_result = baseline_tool_result

print(f"期望结果：{expected_result}")
print(f"基线结果：{baseline_result}")

期望结果：42
基线结果：0


期望结果是 `42`，基线结果却是 `0`，任务没有完成。模型、资源和工具都没有改变，失败来自外层程序混用了三套私有格式。下一章将用统一的 MCP 请求和响应结构替换这条连接链路。

# 6. 定义改进组件 *
## 6.1 定义统一请求格式
当前工业界采用 MCP 时，不让大模型直接拼接协议报文，而是由 Host 内的 Client 把模型决定封装成标准请求。下面固定 JSON-RPC 2.0 的五个字段：协议版本、请求编号、方法名和参数。

In [14]:
# 把一次操作封装成统一的 JSON-RPC 请求
def build_mcp_request(request_id, method, params):
    return {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": method,
        "params": params,
    }


print("统一请求字段：jsonrpc、id、method、params")

统一请求字段：jsonrpc、id、method、params


输出显示所有资源和工具操作都会使用同一套外层结构。`method` 说明要做什么，`params` 保存该操作的数据，`id` 用来关联请求与响应；下一步注册 Server 对外提供的工具。

## 6.2 注册工具
MCP Server 需要同时保存工具说明和实际函数。工具说明供 `tools/list` 返回，实际函数供 `tools/call` 执行；两者使用同一个工具名称连接。

In [15]:
# 定义加法函数，并用同一个名称连接说明与实现
def add(arguments):
    return arguments["a"] + arguments["b"]


mcp_tool_descriptions = [
    {
        "name": "add",
        "description": "计算两个整数的和",
        "inputSchema": {
            "type": "object",
            "properties": {
                "a": {"type": "integer"},
                "b": {"type": "integer"},
            },
            "required": ["a", "b"],
        },
    }
]
mcp_tool_functions = {"add": add}

print(f"已注册工具：{mcp_tool_descriptions[0]['name']}")

已注册工具：add


输出显示 `add` 的说明和实现已经用同一个名称注册。Client 不需要知道函数放在哪里，只需要先取得工具说明，再按名称调用；下一步定义能力发现方法。

### 6.2.1 定义 `tools/list`
`tools/list` 让 Client 从 Server 获取当前可用工具及参数结构，避免在 Client 中重复硬编码工具接口。下面只返回刚才注册的工具说明。

In [16]:
# 返回 Server 当前公开的工具说明
def handle_tools_list(params):
    return {"tools": mcp_tool_descriptions}


print("已定义方法：tools/list")

已定义方法：tools/list


输出说明 `tools/list` 已经连接到工具目录。它只负责公开能力，不执行加法；下一步定义资源读取方法。

## 6.3 定义 `resources/read`
`resources/read` 统一使用 `params.uri` 指定资源。Server 读取数据后，把内容放进标准的 `contents` 列表，Client 不再猜测每个资源服务的私有字段。

In [17]:
mcp_resources = {resource_uri: resource_data}


# 按统一的 uri 参数读取资源
def handle_resources_read(params):
    uri = params["uri"]
    text = json.dumps(mcp_resources[uri])
    return {"contents": [{"uri": uri, "text": text}]}


print("已定义方法：resources/read")

已定义方法：resources/read


输出说明 `resources/read` 已经连接到资源数据。无论资源背后是文件、数据库还是远程服务，Client 看到的都是 `uri → contents`；下一步定义工具调用方法。

## 6.4 定义 `tools/call`
`tools/call` 统一使用 `params.name` 选择工具，使用 `params.arguments` 传入参数。Server 根据注册表找到函数，并把执行结果放进标准的 `content` 列表。

In [18]:
# 根据统一的名称和参数调用已注册工具
def handle_tools_call(params):
    tool_function = mcp_tool_functions[params["name"]]
    result = tool_function(params["arguments"])
    return {"content": [{"type": "text", "text": str(result)}]}


print("已定义方法：tools/call")

已定义方法：tools/call


输出说明 `tools/call` 已经连接到工具实现。Client 只传工具名称和参数，不再了解函数的本地调用方式；最后把三个 MCP 方法接到同一个 Server 入口。

## 6.5 定义统一 Server 入口
MCP Server 根据请求中的 `method` 选择对应处理函数，并把结果放进 JSON-RPC 响应。响应沿用请求的 `id`，因此 Client 可以知道它对应哪一次操作。

In [19]:
mcp_handlers = {
    "tools/list": handle_tools_list,
    "resources/read": handle_resources_read,
    "tools/call": handle_tools_call,
}


# 分发统一请求，并保留同一个请求编号
def mcp_server(request):
    handler = mcp_handlers[request["method"]]
    result = handler(request["params"])
    return {
        "jsonrpc": "2.0",
        "id": request["id"],
        "result": result,
    }


print(f"Server 已连接方法：{list(mcp_handlers)}")

Server 已连接方法：['tools/list', 'resources/read', 'tools/call']


输出显示 Client 和 Server 现在共享三种标准方法及同一套 JSON-RPC 外层结构。第 6 章只完成组件定义，还没有发出 MCP 请求；下一章将复用真实大模型的决定，依次执行能力发现、资源读取和工具调用。

# 7. 展示修复结果 *
## 7.1 发现可用工具
Host 先通过 Client 发送 `tools/list` 请求，取得 Server 当前公开的工具说明。这样，能力名称和参数结构来自 Server，而不是由 Client 另外维护一份私有配置。

In [20]:
# 使用第一个请求编号获取 Server 的工具目录
list_request = build_mcp_request(1, "tools/list", {})
list_response = mcp_server(list_request)
available_tool_name = list_response["result"]["tools"][0]["name"]

print(f"请求：{list_request}")
print(f"Server 公开工具：{available_tool_name}")
print(f"响应 ID：{list_response['id']}")

请求：{'jsonrpc': '2.0', 'id': 1, 'method': 'tools/list', 'params': {}}
Server 公开工具：add
响应 ID：1


输出显示 Server 公开的工具是 `add`，与真实大模型选择的工具一致。响应沿用了请求 ID `1`；下一步用同一套请求结构读取数字资源。

## 7.2 读取资源
Client 把真实大模型选择的资源地址放进标准的 `params.uri`，再调用 `resources/read`。Server 会把资源内容放进标准的 `result.contents`。

In [21]:
# 使用第二个请求编号读取真实大模型选择的资源
read_request = build_mcp_request(
    2,
    "resources/read",
    {"uri": chosen_resource_uri},
)
read_response = mcp_server(read_request)
resource_text = read_response["result"]["contents"][0]["text"]
mcp_resource_result = json.loads(resource_text)

print(f"请求：{read_request}")
print(f"资源返回：{mcp_resource_result}")
print(f"响应 ID：{read_response['id']}")

请求：{'jsonrpc': '2.0', 'id': 2, 'method': 'resources/read', 'params': {'uri': 'numbers://example'}}
资源返回：{'a': 20, 'b': 22}
响应 ID：2


输出显示 `params.uri` 已被 Server 正确读取，资源返回了 `20` 和 `22`，响应 ID 仍为 `2`。下一步把这份资源内容作为标准工具参数发送给 `tools/call`。

## 7.3 调用工具
Client 把真实大模型选择的工具名称放进 `params.name`，把资源内容放进 `params.arguments`。Server 将根据工具注册表找到 `add` 并执行。

In [22]:
# 使用第三个请求编号调用真实大模型选择的工具
call_request = build_mcp_request(
    3,
    "tools/call",
    {
        "name": chosen_tool_name,
        "arguments": mcp_resource_result,
    },
)
call_response = mcp_server(call_request)
result_text = call_response["result"]["content"][0]["text"]
mcp_result = int(result_text)

print(f"请求：{call_request}")
print(f"工具返回：{mcp_result}")
print(f"响应 ID：{call_response['id']}")

请求：{'jsonrpc': '2.0', 'id': 3, 'method': 'tools/call', 'params': {'name': 'add', 'arguments': {'a': 20, 'b': 22}}}
工具返回：42
响应 ID：3


输出显示 `tools/call` 收到了名称 `add` 和参数 `a=20, b=22`，返回结果 `42`，响应 ID 为 `3`。完整任务已经执行，最后集中展示协议数据流和最终结果。

## 7.4 展示最终结果
下面集中打印三次操作的请求 ID、响应 ID、真实大模型决定和最终结果，直观看到每一步数据如何通过统一协议继续向后传递。

In [23]:
# 汇总 MCP 主线中的模型决定、请求编号和最终结果
request_ids = [list_request["id"], read_request["id"], call_request["id"]]
response_ids = [list_response["id"], read_response["id"], call_response["id"]]
improved_result = mcp_result

print(f"模型决定：{decision}")
print(f"请求 ID：{request_ids}")
print(f"响应 ID：{response_ids}")
print(f"期望结果：{expected_result}")
print(f"MCP 结果：{improved_result}")

模型决定：{'resource_uri': 'numbers://example', 'tool_name': 'add'}
请求 ID：[1, 2, 3]
响应 ID：[1, 2, 3]
期望结果：42
MCP 结果：42


请求与响应都使用 ID `[1, 2, 3]`，最终结果与期望结果同为 `42`。真实大模型的决定、资源数据和加法函数都与基线相同；唯一变化是外层程序改用统一的能力发现、资源读取和工具调用协议，因此数据没有在系统交接处丢失。下一章将汇总两种实现的消融对照。

# 8. 汇总消融对照
## 8.1 对比私有格式与 MCP
最后固定同一次真实大模型决定，只比较外层连接方式。下面同时列出共同的 API 成本、两种实现的协议请求数、资源结果、最终结果和任务是否完成。

In [24]:
# 固定模型条件，确保对照实验只改变外层协议
# 汇总任务结果和协议请求数，比较可靠性与额外开销
comparison = {
    "共同条件": {
        "provider": api_record["provider"],
        "model": api_record["model"],
        "模型决定": decision,
        "API 调用次数": 1,
        "input_tokens": api_record["input_tokens"],
        "output_tokens": api_record["output_tokens"],
        "total_tokens": api_record["total_tokens"],
        "latency_ms": api_record["latency_ms"],
        "stop_reason": api_record["stop_reason"],
    },
    "私有格式基线": {
        "协议请求数": 2,
        "资源结果": baseline_resource_result,
        "最终结果": baseline_result,
        "任务成功": baseline_result == expected_result,
    },
    "MCP 改进": {
        "协议请求数": 3,
        "资源结果": mcp_resource_result,
        "请求 ID": request_ids,
        "响应 ID": response_ids,
        "最终结果": improved_result,
        "任务成功": improved_result == expected_result,
    },
}

print(json.dumps(comparison, ensure_ascii=False, indent=2))

{
  "共同条件": {
    "provider": "openai",
    "model": "LongCat-2.0",
    "模型决定": {
      "resource_uri": "numbers://example",
      "tool_name": "add"
    },
    "API 调用次数": 1,
    "input_tokens": 183,
    "output_tokens": 98,
    "total_tokens": 281,
    "latency_ms": 3904,
    "stop_reason": "tool_calls"
  },
  "私有格式基线": {
    "协议请求数": 2,
    "资源结果": null,
    "最终结果": 0,
    "任务成功": false
  },
  "MCP 改进": {
    "协议请求数": 3,
    "资源结果": {
      "a": 20,
      "b": 22
    },
    "请求 ID": [
      1,
      2,
      3
    ],
    "响应 ID": [
      1,
      2,
      3
    ],
    "最终结果": 42,
    "任务成功": true
  }
}


输出显示两种实现共享同一次真实 API 调用，因此模型决定、token 和延迟完全相同。私有格式发送两次请求，却因字段不一致得到 `0`；MCP 多一次 `tools/list` 能力发现请求，随后用统一格式得到 `42`，任务从失败变为成功。

这个对照直接说明了核心问题：模型选对资源和工具并不等于任务能够执行，外层 Harness 还必须把决定可靠地交给不同服务。当前 MCP 的价值正是统一能力发现、资源读取和工具调用的协议边界，减少生产系统中反复编写私有适配代码造成的接口错配。

## 8.2 拓展

### nano 版省略了什么

nano 版只实现 tools/list、resources/read 和 tools/call 的同步内存分发，没有初始化握手、能力协商、通知、sampling、roots、prompts、传输安全、OAuth、取消、进度和协议版本迁移。生产 MCP 还必须验证远程服务器身份与工具元数据，不能把协议互通误当成天然可信。

### 延伸阅读

1. 2024, [Anthropic, Introducing the Model Context Protocol](https://www.anthropic.com/news/model-context-protocol)：MCP 的客户端、服务器与数据源连接模型。
2. 2025, [Model Context Protocol Specification 2025-06-18](https://modelcontextprotocol.io/specification/2025-06-18)：正式生命周期、能力和消息语义。
3. 2025, [OpenAI, New tools and features in the Responses API](https://openai.com/index/new-tools-and-features-in-the-responses-api/)：Remote MCP 在主流 Agent API 中的接入方式。